In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 6.23 The WKB Approximation and the Semiclassical Limit

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VI — Quantum Mechanics",
    number="6.23",
    title="The WKB Approximation and the Semiclassical Limit",
    blurb="Where the wave remembers it was a particle. When the wavelength is "
    "short, the wavefunction's phase is just the classical action, and quantum "
    "mechanics folds back toward the mechanics of Volume II. From this single "
    "idea come the semiclassical wavefunction, the old quantization rule that "
    "counts wavelengths around an orbit, and a formula for tunnelling through a "
    "barrier of any shape — the one Gamow used to explain radioactive decay.",
    difficulty="advanced",
    estimate="175–215 min",
)

## Notebook overview

Almost nothing is exactly solvable, and this movement collects the three great
ways around that fact. Perturbation theory ([§6.21](perturbation-fine-structure.ipynb)) expands around a solvable
Hamiltonian; the variational method ([§6.22](variational-method.ipynb)) guesses a trial state and bounds the
energy from above. The **WKB approximation** is the third and most geometric: it
expands in $\hbar$ itself. When a particle's de Broglie wavelength is short
compared with the scale over which the potential varies, quantum mechanics comes
to *resemble* classical mechanics, and WKB makes the resemblance precise.

The idea is the one Volume II's Hamilton–Jacobi notebook ([§2.10](../02-classical-mechanics/hamilton-jacobi.ipynb)) set up. Write the
wavefunction as $\psi\sim e^{iS/\hbar}$ and expand the phase $S$ in powers of
$\hbar$: the leading term reproduces the classical **action** $\int p\,\mathrm
dx$ and the Hamilton–Jacobi equation, and the next term fixes the amplitude as
$1/\sqrt p$, so the probability density $|\psi|^2\sim1/p\sim1/\text{velocity}$ is
exactly the classical position distribution of [§6.12](harmonic-oscillator.ipynb) — a particle is most likely
found where it moves slowest. The approximation is good where the wavelength
varies slowly and **fails at the classical turning points**, where $p\to0$ and
the wavelength diverges; there the exact solution is an Airy function, and the
**connection formulas** built from it stitch the oscillating and decaying WKB
solutions together. Their $\pi/4$ phase shifts produce the **Bohr–Sommerfeld
quantization** $\oint p\,\mathrm dx=(n+\tfrac12)h$ — the old quantum rule of
[§2.10](../02-classical-mechanics/hamilton-jacobi.ipynb), now with the missing $\tfrac12$ — exact for the harmonic oscillator and
asymptotically exact for any well. In the forbidden region the same machinery
gives the **WKB tunnelling** exponent $T\sim e^{-(2/\hbar)\int|p|\,\mathrm dx}$,
the barrier-of-any-shape generalization of the rectangular result from [§6.13](scattering-tunneling.ipynb), and the
basis of Gamow's theory of alpha decay.

Every result here is checked against something exact: the WKB wavefunction and
spectra against the grid diagonalization of [§6.10](schrodinger-on-a-computer.ipynb) (`scipy.linalg.eigh_tridiagonal`),
and the tunnelling exponent against the exact transmission of a smooth barrier.
We will be honest about where WKB breaks: the turning points, the ground-state
error, and the tunnelling prefactor it gets wrong at leading order.

> **Method specificity.** Each exercise names the exact operation it uses:
> `scipy.integrate.quad` for the action integrals $\oint p\,\mathrm dx$ and
> $\int|p|\,\mathrm dx$; `scipy.optimize.brentq` to solve the Bohr–Sommerfeld
> condition for $E_n$; and `scipy.linalg.eigh_tridiagonal` for the exact
> benchmark spectra. This is the anti-drift discipline of Volume VI.

> **How to read the checks.** Each exercise ends with a validation comparing a
> computed result to an expected fact. A ✗ is a prompt to locate a discrepancy —
> a real error, a convention, or too tight a tolerance — not a verdict that the
> physics is wrong. Passing is strong evidence of correctness, not proof.

## Theory in brief

### The WKB ansatz and the ℏ-expansion

Write the stationary wavefunction as a single exponential, $\psi(x)=e^{iS(x)/\hbar}$,
and substitute into the time-independent Schrödinger equation $-\frac{\hbar^2}{2m}
\psi''+V\psi=E\psi$. With $p(x)=\sqrt{2m[E-V(x)]}$ this gives $(S')^2-i\hbar
S''=p^2$. Expanding $S=S_0+\hbar S_1+\cdots$ in powers of $\hbar$ and matching
orders,

```{math}
:label: eq-wkb-ansatz
(S_0')^2 = p^2 \ \Rightarrow\ S_0 = \pm\!\int p\,\mathrm dx , \qquad
2S_0'S_1' - i S_0'' = 0 \ \Rightarrow\ S_1 = \tfrac{i}{2}\ln p .
```

The leading term is the **classical action** — the Hamilton–Jacobi equation of
[§2.10](../02-classical-mechanics/hamilton-jacobi.ipynb), so classical mechanics is the $\hbar\to0$ limit — and the next term makes
the amplitude $e^{iS_1}=1/\sqrt p$. In the classically **allowed** region ($p$
real) the WKB wavefunction is therefore

```{math}
:label: eq-wkb-amplitude
\psi(x) \approx \frac{1}{\sqrt{p(x)}}\exp\!\left(\pm\frac{i}{\hbar}\!\int
p\,\mathrm dx\right) , \qquad
\psi(x) \approx \frac{1}{\sqrt{|p(x)|}}\exp\!\left(\pm\frac{1}{\hbar}\!\int
|p|\,\mathrm dx\right)
```

in the **forbidden** region ($p$ imaginary), where it grows or decays. The
density $|\psi|^2\sim1/p\sim1/v$ is the classical position distribution of [§6.12](harmonic-oscillator.ipynb):
the amplitude $1/\sqrt p$ is just the statement that the probability flux
$|\psi|^2 v$ is conserved.

### Validity and the turning points

WKB holds when the wavelength $\lambda=2\pi\hbar/p$ varies slowly, $|\mathrm
d\lambda/\mathrm dx|\ll1$ — the potential changes little over a wavelength. It
**fails at the classical turning points** where $E=V$, $p\to0$, and $\lambda\to
\infty$: precisely where the density $1/p$ blows up.

```{math}
:label: eq-wkb-validity
\left|\frac{\mathrm d\lambda}{\mathrm dx}\right|\ll 1
\quad\text{everywhere except near } E=V(x),\ \text{where } p\to0 .
```

Near a turning point the potential is approximately linear, and the exact
solution of the Schrödinger equation is an **Airy function** — oscillatory on the
allowed side, exponentially decaying on the forbidden side. Matching WKB onto the
Airy solution gives the **connection formulas**, which carry a $\pi/4$ phase
shift across each turning point. We state their result and show the Airy behavior
rather than grinding through the asymptotics.

### Bohr–Sommerfeld quantization

For a particle bound between two turning points $x_1<x_2$, requiring the WKB
solution to match consistently through **both** turning points — each
contributing a $\pi/4$ — quantizes the enclosed action:

```{math}
:label: eq-bohr-sommerfeld
\oint p\,\mathrm dx = 2\!\int_{x_1}^{x_2}\! p\,\mathrm dx =
\left(n+\tfrac12\right)h , \qquad n=0,1,2,\dots
```

The $\tfrac12$ is the turning-point correction the old quantum theory ([§2.10](../02-classical-mechanics/hamilton-jacobi.ipynb), which
used $\oint p\,\mathrm dx=nh$) lacked. It is **exact** for the harmonic
oscillator, giving $E_n=(n+\tfrac12)\hbar\omega$, and **asymptotically exact** for
a general well: the error is largest for the ground state and shrinks as $n$ grows
(the semiclassical limit). We solve {eq}`eq-bohr-sommerfeld` for $E_n$ with
`scipy.optimize.brentq` and compare to `scipy.linalg.eigh_tridiagonal`.

### WKB tunnelling and the Gamow factor

In a classically forbidden region under a barrier the WKB solution decays as
$e^{-\frac1\hbar\int|p|\,\mathrm dx}$, so the transmission probability is

```{math}
:label: eq-wkb-tunneling
T \approx \exp\!\left(-\frac{2}{\hbar}\int_{x_1}^{x_2}\!|p|\,\mathrm dx\right)
= \exp\!\left(-\frac{2}{\hbar}\int_{x_1}^{x_2}\!\sqrt{2m[V(x)-E]}\;\mathrm
dx\right) ,
```

the tunnelling exponent for a barrier of **any** shape (the constant-barrier
$e^{-2\kappa a}$ of [§6.13](scattering-tunneling.ipynb) is the special case). It captures the **dominant exponential
suppression**; the prefactor differs at leading order, and we say so honestly.
This is the **Gamow factor** of alpha decay: because the exponent is enormous and
exquisitely sensitive to energy, nuclear half-lives span nanoseconds to billions
of years — the **Geiger–Nuttall law**, $\log(\text{half-life})\propto1/\sqrt E$.

- Reference: Griffiths {cite}`griffiths_qm` (the WKB approximation, connection
  formulas, Bohr–Sommerfeld, tunnelling) and Nolting {cite}`nolting5b`; cross-
  reference Volume II [§2.10](../02-classical-mechanics/hamilton-jacobi.ipynb) (Hamilton–Jacobi, the classical action, action-angle
  and the old quantization), [§6.12](harmonic-oscillator.ipynb) (the classical position distribution, the
  correspondence principle), [§6.13](scattering-tunneling.ipynb) (tunnelling, the constant-barrier limit), [§6.10](schrodinger-on-a-computer.ipynb)
  (the exact benchmark spectra), [§6.21](perturbation-fine-structure.ipynb)/[§6.22](variational-method.ipynb) (the other two approximation methods),
  and forward to [§6.24](time-dependent-perturbation.ipynb) (time-dependent perturbation theory).

---
## Setup

We work in units $\hbar=m=1$ throughout (and, for the oscillator, $\omega=1$), so
an oscillator energy is measured in units of $\hbar\omega$. The reusable core,
each naming the operation it leans on:

- `momentum(x, E, V)`: the classical momentum $p=\sqrt{2m[E-V]}$ (clipped to zero
  in the forbidden region).
- `classical_turning_points(E, V, search)`: the points $V(x)=E$ where $p\to0$, by
  sign changes plus `scipy.optimize.brentq`.
- `exact_spectrum(V, xmax, N, nlev)`: the benchmark eigenvalues of the [§6.10](schrodinger-on-a-computer.ipynb)
  finite-difference Hamiltonian, via `scipy.linalg.eigh_tridiagonal`.
- `bohr_sommerfeld_energy(n, V, search, bracket)`: solves $\int_{x_1}^{x_2}
  p\,\mathrm dx=(n+\tfrac12)\pi\hbar$ for $E_n$ (`scipy.integrate.quad` inside
  `scipy.optimize.brentq`).
- `wkb_wavefunction(x, E, V)`: the allowed-region form $\frac1{\sqrt p}\cos\!\big(
  \frac1\hbar\!\int_{x_1}^{x}p\,\mathrm dx-\frac\pi4\big)$ from
  {eq}`eq-wkb-amplitude` with its connection-formula phase.
- `wkb_tunneling(E, V, search)`: the exponent $e^{-(2/\hbar)\int|p|\,\mathrm dx}$
  of {eq}`eq-wkb-tunneling` between the barrier's turning points.

In [ ]:
import numpy as np
from scipy.integrate import quad, cumulative_trapezoid
from scipy.optimize import brentq
from scipy.linalg import eigh_tridiagonal
from scipy.special import airy
import matplotlib.pyplot as plt

from ecp import draw, validate

HBAR = 1.0
MASS = 1.0


def momentum(x, E, V, m=MASS):
    r"""Classical momentum $p(x)=\sqrt{2m[E-V(x)]}$, clipped to zero where forbidden.

    Parameters
    ----------
    x : float or numpy.ndarray
        Position(s).
    E : float
        Energy.
    V : callable
        Potential ``V(x)``.
    m : float, optional
        Mass.

    Returns
    -------
    float or numpy.ndarray
        The momentum, zero in the classically forbidden region.
    """
    return np.sqrt(np.clip(2.0 * m * (E - V(x)), 0.0, None))


def classical_turning_points(E, V, search, n=4001):
    r"""Turning points $V(x)=E$ via sign changes + ``scipy.optimize.brentq``.

    These are where $p\to0$ and WKB {eq}`eq-wkb-validity` breaks down.

    Parameters
    ----------
    E : float
        Energy.
    V : callable
        Potential.
    search : tuple of float
        ``(xmin, xmax)`` window to scan.
    n : int, optional
        Scan resolution.

    Returns
    -------
    numpy.ndarray
        The sorted turning points.
    """
    xs = np.linspace(search[0], search[1], n)
    f = V(xs) - E
    roots = []
    for i in range(len(xs) - 1):
        if f[i] == 0.0:
            roots.append(xs[i])
        elif f[i] * f[i + 1] < 0.0:
            roots.append(brentq(lambda x: V(x) - E, xs[i], xs[i + 1]))
    return np.array(roots)


def exact_spectrum(V, xmax, N=4000, nlev=8, m=MASS, hbar=HBAR):
    r"""Benchmark spectrum of the §6.10 finite-difference Hamiltonian.

    Builds the $(1,-2,1)/dx^2$ kinetic stencil plus $\mathrm{diag}\,V$ and
    diagonalizes with ``scipy.linalg.eigh_tridiagonal`` (the specialized
    tridiagonal solver of §6.10).

    Parameters
    ----------
    V : callable
        Potential.
    xmax : float
        Half-width of the symmetric grid $[-x_\max, x_\max]$.
    N : int, optional
        Grid points.
    nlev : int, optional
        Number of lowest levels to return.
    m, hbar : float, optional
        Mass and $\hbar$.

    Returns
    -------
    numpy.ndarray
        The ``nlev`` lowest eigenvalues.
    """
    x = np.linspace(-xmax, xmax, N)
    dx = x[1] - x[0]
    diag = hbar**2 / (m * dx**2) + V(x)
    offdiag = -(hbar**2) / (2 * m * dx**2) * np.ones(N - 1)
    return eigh_tridiagonal(
        diag, offdiag, eigvals_only=True, select="i", select_range=(0, nlev - 1)
    )


def bohr_sommerfeld_energy(n, V, search, bracket, m=MASS, hbar=HBAR):
    r"""Solve Bohr–Sommerfeld $\int_{x_1}^{x_2}p\,\mathrm dx=(n+\tfrac12)\pi\hbar$ for $E_n$.

    The action integral is `scipy.integrate.quad`; the root is found with
    `scipy.optimize.brentq`.

    Parameters
    ----------
    n : int
        Quantum number.
    V : callable
        Potential.
    search : tuple of float
        Window for locating turning points.
    bracket : tuple of float
        ``(Elo, Ehi)`` energy bracket for the root.
    m, hbar : float, optional
        Mass and $\hbar$.

    Returns
    -------
    float
        The Bohr–Sommerfeld energy $E_n$.
    """

    def condition(E):
        tp = classical_turning_points(E, V, search)
        x1, x2 = tp[0], tp[-1]
        action, _ = quad(lambda x: momentum(x, E, V, m), x1, x2)
        return action - (n + 0.5) * np.pi * hbar

    return brentq(condition, *bracket)


def wkb_wavefunction(x, E, V, m=MASS, hbar=HBAR):
    r"""WKB wavefunction $\frac1{\sqrt p}\cos(\frac1\hbar\int_{x_1}^x p\,dx-\frac\pi4)$.

    The allowed-region form {eq}`eq-wkb-amplitude` with the connection-formula
    $-\pi/4$ measured from the left turning point $x_1$; ``nan`` outside.

    Parameters
    ----------
    x : numpy.ndarray
        Sorted grid.
    E : float
        Energy.
    V : callable
        Potential.
    m, hbar : float, optional
        Mass and $\hbar$.

    Returns
    -------
    numpy.ndarray
        The (unnormalized) WKB wavefunction, ``nan`` in the forbidden region.
    """
    p = momentum(x, E, V, m)
    allowed = p > 0
    x1 = x[allowed][0]  # left turning point on this grid
    phase = np.zeros_like(x)
    m_ok = x >= x1
    phase[m_ok] = cumulative_trapezoid(p[m_ok], x[m_ok], initial=0.0) / hbar
    psi = np.cos(phase - np.pi / 4.0) / np.sqrt(np.where(allowed, p, np.nan))
    psi[~allowed] = np.nan
    return psi


def wkb_tunneling(E, V, search, m=MASS, hbar=HBAR):
    r"""WKB transmission $T=e^{-(2/\hbar)\int_{x_1}^{x_2}|p|\,dx}$ {eq}`eq-wkb-tunneling`.

    The barrier integral $\int\sqrt{2m[V-E]}\,dx$ between the turning points is
    `scipy.integrate.quad`.

    Parameters
    ----------
    E : float
        Energy (below the barrier top).
    V : callable
        Barrier potential.
    search : tuple of float
        Window for the turning points.
    m, hbar : float, optional
        Mass and $\hbar$.

    Returns
    -------
    float
        The WKB transmission probability.
    """
    tp = classical_turning_points(E, V, search)
    if len(tp) < 2:
        return 1.0
    x1, x2 = tp[0], tp[-1]
    integral, _ = quad(
        lambda x: np.sqrt(np.clip(2 * m * (V(x) - E), 0.0, None)), x1, x2
    )
    return float(np.exp(-2.0 * integral / hbar))

## Exercise 1 — The semiclassical wavefunction

Construct the WKB wavefunction for a highly excited state of the harmonic oscillator
$V(x)=\tfrac12 x^2$ and compare it to the exact eigenfunction. Show that the two agree in the
classically allowed bulk and that WKB breaks down at the turning points.

1. Diagonalize the oscillator with `exact_spectrum`'s machinery
   (`scipy.linalg.eigh_tridiagonal`) to get the exact state $\psi_n$ and energy
   $E_n$ for a high $n$ (take $n=12$).
2. Build the WKB wavefunction with `wkb_wavefunction`: amplitude $1/\sqrt{p}$ and
   phase $\frac1\hbar\int_{x_1}^x p\,\mathrm dx-\frac\pi4$ from
   {eq}`eq-wkb-amplitude`.
3. Fix the single free amplitude by a least-squares match to $\psi_n$ in the bulk
   ($|x|<0.85\,A$, with $A=\sqrt{2E_n}$ the turning point).
4. Overlay the two and mark the turning points; note the agreement away from them
   and the $1/\sqrt p$ blow-up at them.

Cite {eq}`eq-wkb-ansatz`, {eq}`eq-wkb-amplitude`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

The WKB wavefunction must reproduce the exact high-$n$ eigenfunction across the
classically allowed bulk (away from the turning points), confirming
$\psi\sim(1/\sqrt p)e^{i\int p\,\mathrm dx/\hbar}$ as the semiclassical state.

In [ ]:
validate.check(
    rms_bulk < 0.05,
    "the WKB wavefunction matches the exact n=12 eigenfunction in the allowed bulk "
    "(relative RMS < 5%)",
)

## Exercise 2 — The amplitude and the classical distribution

Show that the WKB probability density reproduces the **classical** position distribution of [§6.12](harmonic-oscillator.ipynb):
$|\psi|^2\sim1/p\sim1/\text{velocity}$, so a particle is most likely found where it moves slowest.

1. Form the WKB density $|\psi_{\text{WKB}}|^2\propto1/p(x)$ for the same oscillator
   state, and the smoothed exact density (a running average of $|\psi_n|^2$ over a
   few wavelengths).
2. Write the classical oscillator distribution $P_{\text{cl}}(x)=1/(\pi\sqrt{A^2-x^2})$
   ([§6.12](harmonic-oscillator.ipynb)).
3. Compare all three in the bulk ($|x|<0.7\,A$) and confirm they agree; note the
   divergence at the turning points.
4. Interpret: $1/p$ is $1/\text{velocity}$, the correspondence-principle result.

Cite {eq}`eq-wkb-amplitude`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2

In the bulk the semiclassical density $1/p$ (and the smoothed quantum density)
must match the classical $1/(\pi\sqrt{A^2-x^2})$ to within a few percent —
$|\psi|^2\sim1/\text{velocity}$, the correspondence-principle distribution of [§6.12](harmonic-oscillator.ipynb).

In [ ]:
validate.check(
    abs(ratio - 1.0) < 0.05,
    "the WKB/quantum density matches the classical position distribution in the bulk",
)

## Exercise 3 — Turning points and the connection formulas

Analyze why WKB breaks at a turning point, and state the connection formulas that repair it. Show
the exact solution near a linear turning point — the **Airy function** — passing from oscillation
to exponential decay, and identify the $\pi/4$ phase the connection carries.

1. Confirm the WKB density $1/p$ diverges as $p\to0$ at $E=V$ (the amplitude
   {eq}`eq-wkb-amplitude` is singular there).
2. Near the turning point linearize $V$; the exact solution is the Airy function
   $\mathrm{Ai}(z)$ (`scipy.special.airy`), oscillatory for $z<0$ (allowed) and
   decaying for $z>0$ (forbidden).
3. State the connection formulas: an oscillatory $\frac{2}{\sqrt p}\cos(\frac1\hbar
   \int_x^{x_2}p\,\mathrm dx'-\frac\pi4)$ on the allowed side joins a decaying
   $\frac{1}{\sqrt{|p|}}e^{-\frac1\hbar\int_{x_2}^x|p|\,\mathrm dx'}$ on the
   forbidden side. The $\pi/4$ is the fingerprint of the Airy match.
4. Note that the turning point is where the semiclassical picture needs genuinely
   quantum repair.

Cite {eq}`eq-wkb-validity`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

The Airy function must show the two regimes the connection formulas join:
oscillatory for $z<0$ (several sign changes) and monotone decay for $z>0$
($\mathrm{Ai}(z)\to0$ without oscillating). That contrast is the turning-point
physics WKB alone cannot supply.

In [ ]:
sign_changes = np.sum(np.diff(np.sign(Ai[z < 0])) != 0)
decays = Ai[z > 0][-1] < Ai[z > 0][0] and np.all(Ai[z > 3.0] < 0.05)
validate.check(
    sign_changes >= 3 and decays,
    "Airy Ai(z) oscillates for z<0 and decays for z>0 (the connection-formula regimes)",
)

## Exercise 4 — Bohr–Sommerfeld: the oscillator (exact)

Apply the Bohr–Sommerfeld condition {eq}`eq-bohr-sommerfeld` to the harmonic oscillator and show
it is **exact**: $E_n=(n+\tfrac12)\hbar\omega$, the $\tfrac12$ now present (unlike the old quantum
theory of [§2.10](../02-classical-mechanics/hamilton-jacobi.ipynb)).

1. For each $n$, form the action $\int_{x_1}^{x_2}p\,\mathrm dx$ as a function of
   $E$ (`scipy.integrate.quad` between the turning points).
2. Solve $\int p\,\mathrm dx=(n+\tfrac12)\pi\hbar$ for $E_n$ with
   `bohr_sommerfeld_energy` (`scipy.optimize.brentq`).
3. Confirm $E_n=(n+\tfrac12)$ to numerical tolerance.
4. Visualize the root-finding: the action curve crossing the quantized levels.

Cite {eq}`eq-bohr-sommerfeld`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4

For the oscillator Bohr–Sommerfeld is exact: the roots of {eq}`eq-bohr-sommerfeld`
must equal $(n+\tfrac12)\hbar\omega$ to the tolerance of the quadrature and root
finder.

In [ ]:
validate.close(
    E_bs_osc,
    n_vals + 0.5,
    "Bohr–Sommerfeld is exact for the oscillator, E_n = (n+1/2)ℏω",
    rtol=1e-4,
    atol=1e-4,
)

## Exercise 5 — Bohr–Sommerfeld for a general well

Apply Bohr–Sommerfeld to the **quartic** well $V(x)=x^4$, which has no closed-form spectrum, and
compare to exact diagonalization. Show that WKB is **asymptotically exact**: the error is largest
for the ground state and shrinks as $n$ grows.

1. Compute the exact quartic spectrum with `exact_spectrum`
   (`scipy.linalg.eigh_tridiagonal`).
2. For each $n$, solve $\int p\,\mathrm dx=(n+\tfrac12)\pi\hbar$ with
   `bohr_sommerfeld_energy` (`scipy.optimize.brentq`, action by
   `scipy.integrate.quad`).
3. Tabulate the relative error $|E_n^{\text{WKB}}-E_n^{\text{exact}}|/E_n^{\text{exact}}$.
4. Show it falls monotonically with $n$ — the semiclassical limit — and plot both
   the levels and the error.

Cite {eq}`eq-bohr-sommerfeld`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

WKB must approach the exact quartic spectrum from the semiclassical direction:
the relative error is largest at $n=0$ and decreases with $n$, dropping below 1%.

In [ ]:
validate.check(
    rel_err[0] > rel_err[-1] and np.all(np.diff(rel_err) < 0) and rel_err[-1] < 0.01,
    "Bohr–Sommerfeld quartic error is largest at n=0 and shrinks monotonically below "
    "1% (the semiclassical limit)",
)

## Exercise 6 — WKB tunnelling vs exact transmission

Compute the WKB tunnelling probability through a smooth barrier and compare to the **exact**
transmission. Confirm that the WKB exponent captures the dominant exponential suppression,
honestly noting that the prefactor differs.

1. Take the Eckart barrier $V(x)=V_0\,\mathrm{sech}^2(x/a)$ (a smooth barrier with a
   known exact transmission), with $V_0=6$, $a=1$.
2. For each $E<V_0$, find the turning points where $V=E$
   (`scipy.optimize.brentq`) and compute the exponent $(2/\hbar)\int|p|\,\mathrm dx$
   with `wkb_tunneling` (`scipy.integrate.quad`).
3. Evaluate $T_{\text{WKB}}=e^{-(2/\hbar)\int|p|\,\mathrm dx}$ and the exact Eckart
   transmission $T(E)$.
4. Compare on a log axis; confirm the exponents agree (the slopes) while the
   prefactor differs — the constant-barrier $e^{-2\kappa a}$ of [§6.13](scattering-tunneling.ipynb) is the special
   case.

Cite {eq}`eq-wkb-tunneling`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

In the deep-tunnelling regime the WKB and exact transmissions must agree in the
**exponent** — the physically dominant quantity — to a few percent, even though
their prefactors differ: $T\sim e^{-(2/\hbar)\int|p|\,\mathrm dx}$ is the WKB
tunnelling formula.

In [ ]:
validate.check(
    np.max(log_gap) < 0.05,
    "the WKB tunnelling exponent captures the exact transmission's exponential "
    "suppression (log agrees to <5% deep under the barrier)",
)

## Exercise 7 — Alpha decay and the Geiger–Nuttall law *(student)*

Use the WKB tunnelling exponent to explain the enormous range of alpha-decay half-lives. Model the
alpha particle escaping through the nuclear Coulomb barrier, compute the Gamow exponent, and show
that $\log T$ is linear in $1/\sqrt E$ — the Geiger–Nuttall law — so a small change in energy
changes the rate by orders of magnitude.

1. Model the barrier outside the nucleus as Coulomb, $V(r)=C/r$ for $r>R$ (a
   nuclear radius $R$), in scaled units.
2. For a decay energy $E$, the outer turning point is $b=C/E$; compute the Gamow
   exponent $\gamma(E)=\frac{2}{\hbar}\int_R^{b}\sqrt{2m[C/r-E]}\,\mathrm dr$ with
   `scipy.integrate.quad`.
3. Show $\ln T=-\gamma$ is linear in $1/\sqrt E$ (fit it) and spans many orders of
   magnitude across a modest energy range.
4. Connect to Geiger–Nuttall: $\log(\text{half-life})\propto1/\sqrt E$.

Cite {eq}`eq-wkb-tunneling`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

The Gamow exponent must make $\ln T$ linear in $1/\sqrt E$ (the Geiger–Nuttall law)
and span many orders of magnitude across a modest energy range — the origin of the
vast spread of alpha-decay half-lives.

In [ ]:
validate.check(
    r2 > 0.999 and np.exp(np.ptp(ln_T)) > 1e4,
    "the Coulomb-barrier Gamow exponent gives ln T ∝ 1/√E (Geiger–Nuttall) spanning "
    ">4 orders of magnitude",
)

## Exercise 8 — (Synthesis) The seam between two mechanics

The WKB approximation is where the wave remembers it was a particle. We wrote the
wavefunction as $\psi\sim e^{iS/\hbar}$ and found its phase to be nothing but the
classical action of Volume II; in the limit of a vanishing wavelength the
Schrödinger equation became the Hamilton–Jacobi equation of [§2.10](../02-classical-mechanics/hamilton-jacobi.ipynb), and classical
mechanics reappeared as the $\hbar\to0$ shadow of quantum mechanics. From that one
idea came the semiclassical wavefunction, the density $1/p$ that is the classical
distribution of [§6.12](harmonic-oscillator.ipynb), the old quantization rule of [§2.10](../02-classical-mechanics/hamilton-jacobi.ipynb) — now with its missing
$\tfrac12$ restored by the turning-point phases — and a tunnelling formula good for
a barrier of any shape, the formula that turned the wild scatter of radioactive
half-lives into a single exponential.

With WKB we hold the third and last of the great approximation methods. Faced with
a problem no one can solve exactly, one can expand around a solvable Hamiltonian
(perturbation theory, [§6.21](perturbation-fine-structure.ipynb)), guess a trial state and bound the energy
(variational, [§6.22](variational-method.ipynb)), or shrink the wavelength and let mechanics go classical
(semiclassical, WKB). Perturbation theory needs a nearby solvable neighbour;
the variational method needs a good guess; WKB needs a short wavelength. Between
them they cover most of what the exactly-solvable models of Movements I–IV cannot
reach.

It is worth pausing on the strangeness of the bridge. Bohr counted wavelengths
around an orbit and got the hydrogen spectrum nearly right; Hamilton, a century
before anyone suspected the electron was a wave, wrote mechanics in the language of
phases and actions. WKB shows why both were reaching the same truth from opposite
sides: the action *is* a phase, and quantization is what happens when a wave has to
close on itself. The next notebook ([§6.24](time-dependent-perturbation.ipynb)) puts time back in and asks how a
perturbation that switches on drives transitions between states — Fermi's golden
rule, and the theory of how atoms absorb and emit light.

## Notebook summary

- **The WKB ansatz** {eq}`eq-wkb-ansatz`: $\psi\sim e^{iS/\hbar}$ expanded in
  $\hbar$ gives the classical action at leading order (Hamilton–Jacobi, [§2.10](../02-classical-mechanics/hamilton-jacobi.ipynb)) and
  the amplitude $1/\sqrt p$ at next order — verified against the exact $n=12$
  oscillator state to under 2% in the bulk.
- **The classical distribution** {eq}`eq-wkb-amplitude`: $|\psi|^2\sim1/p\sim1/v$,
  matching the classical $1/(\pi\sqrt{A^2-x^2})$ of [§6.12](harmonic-oscillator.ipynb).
- **Turning points**: the $1/\sqrt p$ amplitude diverges where $p\to0$; the exact
  Airy solution (`scipy.special.airy`) and its $\pi/4$-carrying connection formulas
  repair the match.
- **Bohr–Sommerfeld** {eq}`eq-bohr-sommerfeld`: $\oint p\,\mathrm dx=(n+\tfrac12)h$
  is exact for the oscillator and asymptotically exact for the quartic (error
  $18\%\to0.1\%$ as $n$ grows), by `scipy.optimize.brentq` vs
  `scipy.linalg.eigh_tridiagonal`.
- **WKB tunnelling** {eq}`eq-wkb-tunneling`: $T\sim e^{-(2/\hbar)\int|p|\,\mathrm dx}$
  captures the exact Eckart-barrier exponent (prefactor aside), and its Coulomb-
  barrier form gives Gamow's Geiger–Nuttall law ($\ln T\propto1/\sqrt E$).

## Outlook

- **Time-dependent perturbation theory and Fermi's golden rule**: transitions,
  absorption, and emission ([§6.24](time-dependent-perturbation.ipynb)).
- **The path integral and its semiclassical limit**; instantons and tunnelling as
  least-action paths in imaginary time (horizons).
- **The Maslov index and higher-order WKB corrections** — systematically improving
  the $\hbar$-expansion (horizons).
- Cross-reference Volume II [§2.10](../02-classical-mechanics/hamilton-jacobi.ipynb) (Hamilton–Jacobi, the action, action-angle and the
  old quantization), [§6.12](harmonic-oscillator.ipynb) (the classical distribution), [§6.13](scattering-tunneling.ipynb) (tunnelling), [§6.10](schrodinger-on-a-computer.ipynb)
  (exact spectra), [§6.21](perturbation-fine-structure.ipynb)/[§6.22](variational-method.ipynb) (the other approximation methods), forward to [§6.24](time-dependent-perturbation.ipynb).

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()